# Construir un módulo de TethysDash — el gráfico de ejemplo

Todo lo que se dibuja en un tablero de TethysDash lo produce un **módulo**: un
paquete de Python instalable que la aplicación descubre por su cuenta. Este
cuaderno construye desde cero el más simple que sirve de algo —un gráfico de
líneas— y después le agrega un argumento y una barra de progreso.

La pregunta que se responde es:

> **¿Cuál es la mínima cantidad de Python que pone un gráfico nuevo en el selector
> de visualizaciones?**

La respuesta es una clase con cuatro atributos y un método. Todo lo demás es
opcional.

**Qué llevarse de este cuaderno**

- qué está obligado a devolver `run()`, y cómo comprobarlo usted mismo
- cómo declarar `args` genera controles de la interfaz sin escribir un formulario
- por qué `get_arg()` es más seguro que leer el argumento desde `self`
- qué significa registrar un módulo, y por qué no requiere modificar TethysDash

**Flujo de trabajo**

1. Obtener los datos y construir el gráfico con plotly, como en cualquier parte
2. Ver qué produce `to_json()`: eso *es* el valor de retorno del módulo
3. Envolverlo en una clase y ejecutarlo aquí, sin servidor
4. Agregar un argumento y ver de dónde sale el control de la interfaz
5. Agregar mensajes de progreso
6. Registrarlo para que la aplicación lo encuentre

Un módulo real importa su clase base de TethysDash:

```python
from tethysapp.tethysdash.plugin_helpers import TethysDashPlugin
```

Este cuaderno define en su lugar una **clase sustituta** con el mismo contrato,
para que todas las celdas funcionen en una computadora que solo tenga plotly
instalado. Esa única línea de importación es la única diferencia entre lo que hay
aquí y el archivo de un módulo real.

In [1]:
!pip install plotly


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json

import plotly.express as px
import plotly.graph_objects as go

VALID_TYPES = ["plotly", "table", "image", "card", "text", "variable_input",
               "map", "map_layer", "custom", "imageCollection"]

class TethysDashPlugin:
    """Sustituto didáctico de la clase base real, con el mismo contrato.

    Reproduce lo que hace la real al construirse: valida los cuatro atributos
    obligatorios, rechaza un `type` desconocido y expone los argumentos
    recibidos tanto por `get_arg()` como por acceso a atributos.

    Aquí `send_update()` solo imprime. La real publica por un WebSocket y
    necesita el contexto de la solicitud que la aplicación adjunta al invocar un
    módulo, así que no se puede llamar fuera de un servidor en ejecución. Por eso
    este cuaderno usa un sustituto en lugar de importar la clase real.

    Los mensajes de error se dejan en inglés a propósito: son exactamente los que
    lanza TethysDash, y conviene reconocerlos cuando aparezcan en el servidor.
    """

    args = {}

    def __init__(self, **kwargs):
        for attr in ("name", "type", "label", "group"):
            if getattr(self, attr, None) in (None, ""):
                raise ValueError(f"Plugin must have a {attr} attribute defined.")
        if self.type not in VALID_TYPES:
            raise ValueError(
                f"Plugin type '{self.type}' is not valid. "
                f"Must be one of: {', '.join(VALID_TYPES)}"
            )
        self.received_args = dict(kwargs)
        for key, value in kwargs.items():
            setattr(self, key, value)

    def get_arg(self, name, default=None):
        return self.received_args.get(name, default)

    def send_update(self, message, percentage_complete=None, layer_id=None):
        pct = "" if percentage_complete is None else f" [{percentage_complete}%]"
        print(f"  progreso{pct}: {message}")

print(f"sustituto listo | {len(VALID_TYPES)} tipos de módulo válidos")

sustituto listo | 10 tipos de módulo válidos


## 1. El gráfico, hecho de la forma habitual

Nada de este paso es específico de TethysDash. Es el gráfico que usted escribiría
en cualquier cuaderno, y ahí está el punto: un módulo es el código que ya tiene,
envuelto para que la aplicación pueda llamarlo.

In [3]:
df = px.data.gapminder()
print(f"{len(df):,} filas, {df.country.nunique()} países, "
      f"{df.year.min()}-{df.year.max()}")
df.head(3)

1,704 filas, 142 países, 1952-2007


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4


In [4]:
asia = df.query("continent == 'Asia'")
fig = px.line(asia, x="year", y="lifeExp", color="country", symbol="country")
fig

## 2. Qué tiene que devolver `run()`

Un módulo declara `type = "plotly"`, y esa elección determina la forma de su valor
de retorno. Para `plotly` el contrato es la figura como un **diccionario común**:
exactamente lo que produce `to_json()`.

Mire las claves, no el contenido:

In [5]:
payload = json.loads(fig.to_json())

print("claves de primer nivel:", list(payload))
print(f"  data:   {len(payload['data'])} trazas")
print(f"  layout: {len(payload['layout'])} claves -> {list(payload['layout'])[:6]}...")
print(f"\ntamaño serializado: {len(fig.to_json()) / 1024:.0f} KB")

claves de primer nivel: ['data', 'layout']
  data:   33 trazas
  layout: 5 claves -> ['template', 'xaxis', 'yaxis', 'legend', 'margin']...

tamaño serializado: 25 KB


Ese diccionario es toda la interfaz. El módulo no dibuja nada y no toca el
navegador: devuelve datos, y el frontend los renderiza.

Lo que significa que el contrato se puede comprobar sin servidor. Vuelva a
convertir el diccionario en una figura: si se dibuja, un tablero real dibujaría lo
mismo.

In [6]:
go.Figure(payload)

## 3. El módulo mínimo

Cuatro atributos obligatorios y un método:

| atributo | qué hace |
|---|---|
| `name` | el nombre de instalación y del *driver*; debe coincidir con el punto de entrada |
| `group` | agrupa el módulo en el selector de visualizaciones |
| `label` | el nombre que se muestra en la aplicación |
| `type` | elige el renderizador y, por lo tanto, dicta qué devuelve `run()` |

Si falta cualquiera de ellos, la construcción falla, así que un módulo mal
definido falla de forma ruidosa en lugar de aparecer a medias en la aplicación.

In [7]:
class PlotExample(TethysDashPlugin):
    name = "plot_example"
    group = "Example"
    label = "Example Plot"
    type = "plotly"

    def run(self):
        frame = px.data.gapminder().query("continent == 'Asia'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        return json.loads(figure.to_json())


plugin = PlotExample()
result = plugin.run()
print(f"run() devolvió {type(result).__name__} con las claves {list(result)}")
print(f"  {len(result['data'])} trazas")

run() devolvió dict con las claves ['data', 'layout']
  33 trazas


Instanciar la clase y llamar a `run()` es también la forma de probar un módulo:
sin servidor, sin tablero y sin navegador. Si `run()` devuelve la forma correcta,
la visualización funciona.

In [8]:
# Qué pasa cuando falta un atributo obligatorio.
class Broken(TethysDashPlugin):
    name = "broken"
    type = "plotly"
    label = "Broken"
    # falta group

try:
    Broken()
except ValueError as err:
    print(f"ValueError: {err}")

ValueError: Plugin must have a group attribute defined.


## 4. Agregar un argumento

Declarar `args` es lo que produce los controles de la interfaz. Quien arma el
tablero nunca ve un formulario escrito por usted: TethysDash genera el control a
partir del tipo que se declara, y la etiqueta a partir del nombre del argumento.

`{"continent": "text"}` se convierte en un campo de texto etiquetado
**Continent**.

In [9]:
class PlotByContinent(TethysDashPlugin):
    name = "plot_by_continent"
    group = "Example"
    label = "Plot by Continent"
    type = "plotly"
    args = {"continent": "text"}      # -> un campo de texto, etiquetado "Continent"

    def run(self):
        continent = self.get_arg("continent", "Asia")
        frame = px.data.gapminder().query(f"continent == '{continent}'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        figure.update_layout(title=f"Esperanza de vida — {continent}")
        return json.loads(figure.to_json())


# La aplicación pasa los valores configurados; aquí los pasamos a mano.
for continent in ("Europe", "Africa"):
    out = PlotByContinent(continent=continent).run()
    print(f"{continent:<8} {len(out['data']):>2} trazas  "
          f"título={out['layout']['title']['text']!r}")

Europe   30 trazas  título='Esperanza de vida — Europe'


Africa   52 trazas  título='Esperanza de vida — Africa'


In [10]:
go.Figure(PlotByContinent(continent="Europe").run())

### Lea los argumentos con `get_arg()`, no desde `self`

El ejemplo de las diapositivas usa `self.continent` y, para un nombre simple, eso
funciona: el marco de trabajo define cada argumento recibido como atributo. Pero
**conviene preferir `get_arg()`**, por dos razones:

- **Los argumentos anidados tienen nombres con puntos**, como
  `transect_location.location`. Python no puede resolver un atributo con puntos,
  así que `self.transect_location.location` falla, y en el peor de los casos
  falla en silencio.
- **`get_arg()` acepta un valor por defecto.** Un argumento que quedó vacío
  simplemente no existe, así que el acceso por atributo lanza `AttributeError`
  mientras que `get_arg("continent", "Asia")` sigue adelante.

Además, `args` no puede usar los nombres de las propiedades del propio módulo:
`type`, `label`, `group`, `tags` y las demás están reservadas, y chocar con una de
ellas hace fallar la construcción.

## 5. Mensajes de progreso

Un módulo que tarda unos segundos debería decirlo. `send_update()` envía un
mensaje —y opcionalmente un porcentaje— al elemento del tablero por un WebSocket,
lo que convierte un recuadro en blanco en una barra de progreso.

Aquí el sustituto solo imprime, para que se vea la secuencia.

Una nota práctica: el `send_update()` real necesita el contexto de la solicitud que
la aplicación adjunta al llamar a un módulo, así que solo funciona dentro de un
servidor en ejecución. Llamarlo desde un script lanza `AttributeError`. En la
práctica no es un problema: solo significa que el reporte de progreso es la única
parte de un módulo que no se puede ejercitar fuera de la aplicación.

In [11]:
class PlotWithProgress(TethysDashPlugin):
    name = "plot_with_progress"
    group = "Example"
    label = "Plot with Progress"
    type = "plotly"
    args = {"continent": "text"}

    def run(self):
        continent = self.get_arg("continent", "Asia")

        self.send_update("Obteniendo datos de plotly", percentage_complete=25)
        frame = px.data.gapminder().query(f"continent == '{continent}'")

        self.send_update("Graficando los datos", percentage_complete=75)
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")

        self.send_update("Listo", percentage_complete=100)
        return json.loads(figure.to_json())


print("ejecutando PlotWithProgress(continent='Oceania')")
out = PlotWithProgress(continent="Oceania").run()
print(f"-> {len(out['data'])} trazas")

ejecutando PlotWithProgress(continent='Oceania')
  progreso [25%]: Obteniendo datos de plotly
  progreso [75%]: Graficando los datos
  progreso [100%]: Listo
-> 2 trazas


Para los módulos de inundación de los ejercicios 2 y 3 esto no es cosmético.
Muestrear la profundidad sobre 5,000 edificios tarda unos segundos y, sin
progreso, el recuadro parece averiado en lugar de ocupado.

## 6. Registrarlo

El módulo ya existe, pero la aplicación todavía no lo ve. Registrarlo es un solo
punto de entrada en el `pyproject.toml` del paquete. TethysDash no se modifica
nunca.

```toml
[project.entry-points."intake.drivers"]
plot_example = "my_plugin.source:PlotExample"
```

Y después:

```bash
pip install .        # en el entorno donde corre TethysDash
```

Reinicie la aplicación y el módulo aparece en el desplegable
**Visualization Type**, bajo el `group` que haya declarado. Ocurren seis cosas, y
ninguna dentro de TethysDash:

1. Se declara el punto de entrada en `pyproject.toml` (o `setup.py`)
2. `pip install` coloca el paquete en el entorno de TethysDash
3. Intake lo registra automáticamente: `open_plot_example` queda disponible
4. El módulo aparece en el selector de visualizaciones
5. Una carpeta `static/` con miniaturas lo hace reconocible entre muchos
6. Probarlo es instanciar la clase y llamar a `run()`, como arriba

Vale la pena decir en voz alta la consecuencia institucional: **instalar un módulo
es instalar un paquete de Python.** No requiere un procedimiento nuevo, ni tocar el
código de TethysDash, ni nada que impida actualizar la plataforma más adelante.

### Las propiedades opcionales

| propiedad | por defecto | qué hace |
|---|---|---|
| `args` | `{}` | esquema de argumentos → controles generados |
| `tags` | `[]` | búsqueda y descubrimiento |
| `description` | `""` | se muestra a los usuarios junto a la selección |
| `restricted` | `False` | limita el módulo a usuarios con permiso |
| `loading_icon` | `True` | indicador de carga mientras el módulo se ejecuta |
| `attribution` | `""` | crédito de la fuente de datos dibujado en el elemento |
| `dynamic_map_layer` | `False` | habilita `fetch_features()` para capas de mapa dinámicas |

Dos importan en un contexto institucional: `restricted`, que deja el módulo detrás
de permisos, y `attribution`, que imprime automáticamente el crédito de la fuente
de datos, algo que suele ser un requisito formal al publicar datos de terceros.

### Cosas para probar

- Cambie `type` por algo inválido y vea el error. Después pruebe con `"table"` y
  haga que `run()` devuelva `{"title": ..., "data": [ ... ]}` en su lugar: la misma
  clase, otro renderizador.
- Declare `args = {"continent": "text", "year": "number"}` y filtre por los dos.
  ¿Qué mostraría la interfaz?
- Pruebe con `args = {"type": "text"}` y lea el error. ¿Por qué está reservado ese
  nombre?
- Quite el valor por defecto de `get_arg` e instancie sin `continent`. Compare la
  falla con lo que habría hecho `self.continent`.
- El gráfico se vuelve a dibujar por completo en cada llamada. ¿Qué partes de
  `run()` guardaría en caché si los datos vinieran de una fuente lenta y no de
  plotly?